# Phase 7 Benchmark — Analysis Notebook (v2)

Visualizations of `benchmark/results/benchmark.csv` across four parser conditions (MinerU, GROBID, Docling, Router) under the recall-aware v2 schema from plan 07-02.5.

See `benchmark/FINDINGS.md` for the formal report.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Run from project root or from benchmark/notebook/
HERE = os.getcwd()
CSV_PATH = os.path.join('..', 'results', 'benchmark.csv') \
    if os.path.basename(HERE) == 'notebook' \
    else os.path.join('benchmark', 'results', 'benchmark.csv')

df = pd.read_csv(CSV_PATH)
METRIC_COLS = ['heading_precision', 'heading_recall', 'heading_f1',
               'hierarchy_f1', 'coherent_section_pct', 'table_presence',
               'table_structural_completeness', 'body_token_count',
               'sec_per_doc', 'figure_count_parser', 'formula_count_parser',
               'reference_count_parser']
for c in METRIC_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['error'] = df['error'].fillna('').astype(str)
ok = df[df['error'] == '']
COND = ['mineru', 'grobid', 'docling', 'router']
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
print(f'Total rows: {len(df)}; non-errored: {len(ok)}; '
      f'conditions: {sorted(df.condition.unique())}')
df.head(3)

## 1. Heading F1 by Condition (bar chart)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
means = ok.groupby('condition')['heading_f1'].mean().reindex(COND)
means.plot(kind='bar', ax=ax, color=COLORS)
ax.set_ylabel('Heading F1 (mean)')
ax.set_title('Heading F1 by Parser Condition (recall-aware v2)')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Heading Precision vs Recall (grouped bar)

In [ ]:
pr = ok.groupby('condition')[['heading_precision', 'heading_recall']].mean().reindex(COND)
ax = pr.plot(kind='bar', figsize=(8, 4))
ax.set_ylabel('Mean')
ax.set_title('Heading Precision vs Recall by Condition')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Hierarchy F1 by Condition (router differentiator)

Per plan 07-02.5, only the router applies `_apply_dot_count_hierarchy`. Standalone parsers return 0.0 by construction.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
h = ok.groupby('condition')['hierarchy_f1'].mean().reindex(COND)
h.plot(kind='bar', ax=ax, color=COLORS)
ax.set_ylabel('Hierarchy F1 (mean)')
ax.set_title('Hierarchy F1 by Condition (router-only builder)')
ax.set_ylim(0, max(0.1, h.max() * 1.2))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Table Quality — Presence vs Structural Completeness (scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for cond, color in zip(COND, COLORS):
    sub = ok[ok['condition'] == cond]
    jitter = 0.02 * (hash(cond) % 5 - 2)
    ax.scatter(sub['table_presence'] + jitter,
               sub['table_structural_completeness'],
               alpha=0.4, label=cond, color=color, s=25)
ax.set_xlabel('Table Presence (0 or 1, jittered)')
ax.set_ylabel('Table Structural Completeness (0.0–1.0)')
ax.set_title('Table Extraction Quality')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Heading F1 Distribution by Condition (box plot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
data = [ok[ok['condition'] == c]['heading_f1'].dropna().values for c in COND]
ax.boxplot(data, labels=['MinerU', 'GROBID', 'Docling', 'Router'])
ax.set_ylabel('Heading F1')
ax.set_title('Heading F1 Distribution by Parser Condition')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Content Richness — Body Token Count per Condition

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
data = [ok[ok['condition'] == c]['body_token_count'].dropna().values for c in COND]
ax.boxplot(data, labels=['MinerU', 'GROBID', 'Docling', 'Router'])
ax.set_ylabel('Body token count')
ax.set_title('Body Token Count Distribution by Condition (tiktoken cl100k)')
plt.tight_layout()
plt.show()

## 7. Reference Count — Parser vs Ground Truth (bar)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
parser_med = ok.groupby('condition')['reference_count_parser'].median().reindex(COND)
gt_med = ok.groupby('condition')['reference_count_gt'].median().reindex(COND)
x = range(len(COND))
ax.bar([i - 0.2 for i in x], parser_med.values, width=0.4, label='parser', color='#1f77b4')
ax.bar([i + 0.2 for i in x], gt_med.values, width=0.4, label='GT', color='#d62728')
ax.set_xticks(list(x))
ax.set_xticklabels(['MinerU', 'GROBID', 'Docling', 'Router'])
ax.set_ylabel('Reference count (median)')
ax.set_title('Reference Extraction — Parser vs GT (median per paper)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Single- vs Two-Column Breakdown

The 07-02 corpus contains 0 two-column papers, so the two-column bars will be empty. This limitation is documented in FINDINGS.md.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes,
                             ['heading_f1', 'coherent_section_pct'],
                             ['Heading F1', 'Coherent Section %']):
    pivot = ok.groupby(['condition', 'column_layout'])[metric].mean().unstack('column_layout')
    pivot = pivot.reindex(COND)
    pivot.plot(kind='bar', ax=ax)
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Per-Condition Error Rate

In [ ]:
error_rates = df.assign(is_error=(df['error'] != '')).groupby('condition')['is_error'].mean()
error_rates = error_rates.reindex(COND)
error_rates.plot(kind='bar', figsize=(7, 4), color='#d62728')
plt.ylabel('Error rate (fraction)')
plt.title('Per-Condition Error Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()